# Train S.C.R.A.M as an art detector (Google Colab)

This notebook teaches the detector to tell **human-made art** from **AI-generated art**:

1. downloads the art datasets (AniPrO, Kaggle Human-AI Artwork, Kaggle Real & Fake Art, and ART500K if you have it)
2. adds them to the training folder (human art → `real`, each dataset's AI art → its own class)
3. trains the ViT-B/16 one AI class at a time with **experience replay**, so it doesn't forget earlier classes
4. evaluates on held-out test images and shows the metrics
5. lets you try it on your own images
6. saves everything to your Google Drive

**Before you start**

- **Use a GPU runtime:** *Runtime → Change runtime type → T4 GPU* (L4 or A100 is 2–4× faster if you have Colab Pro).
- **Put the project on Drive:** upload `AI-detector-model-improved.zip` to `My Drive/scram/`, or fill in `GITHUB_URL` instead.
- **Add your Kaggle API token:** in the 🔑 *Secrets* panel on the left, add `KAGGLE_USERNAME` and `KAGGLE_KEY` and turn on notebook access. Get a token at kaggle.com → *Settings → API → Create New Token*.
- Optionally add `HF_TOKEN` (huggingface.co → *Settings → Access Tokens*) if AniPrO asks you to log in.

**Time and disk.** With the default settings (6,000 images per class, 3 epochs) expect roughly 2–4 hours on a T4. Training saves to Drive after each AI class. If Colab disconnects, run all the cells again and it carries on from the last finished class. Turn on `QUICK_TEST` for a ~20-minute run on a small sample first, to check everything works.

**Licences.** AniPrO is CC BY-NC 4.0 and ART500K is for non-commercial research only. Check each Kaggle dataset's licence before any commercial use.

## 1. Settings

In [ ]:
#@title Settings { display-mode: "form" }
#@markdown **Where things live on your Google Drive**
DRIVE_DIR = "/content/drive/MyDrive/scram"  #@param {type:"string"}
PROJECT_ZIP = "AI-detector-model-improved.zip"  #@param {type:"string"}
#@markdown Leave empty to use the zip above. Otherwise, e.g. https://github.com/you/AI-detector-model
GITHUB_URL = ""  #@param {type:"string"}

#@markdown **What to train**
#@markdown - `art_only`: a new model that knows human art vs the AI art in these datasets
#@markdown - `continue`: extend your existing photo model (needs `base_model/` and the original dataset zip on Drive)
MODE = "art_only"  #@param ["art_only", "continue"]
BASE_MODEL_DIR = "base_model"  #@param {type:"string"}
BASE_DATASET_ZIP = "defactify_dataset.zip"  #@param {type:"string"}

#@markdown **Datasets**
USE_ANIPRO = True  #@param {type:"boolean"}
USE_HUMAN_AI_ARTWORK = True  #@param {type:"boolean"}
USE_REAL_FAKE_ART = True  #@param {type:"boolean"}
#@markdown Path on Drive (relative to DRIVE_DIR) to an ART500K zip you downloaded yourself. Leave empty to skip.
ART500K_ZIP = ""  #@param {type:"string"}

#@markdown **Training**
MAX_PER_CLASS = 6000  #@param {type:"integer"}
EPOCHS_PER_EXPERIENCE = 3  #@param {type:"integer"}
BATCH_SIZE = 32  #@param {type:"integer"}
REPLAY_IMAGES_PER_GENERATOR = 2000  #@param {type:"integer"}
#@markdown Small sample and 1 epoch, to check the whole pipeline in ~20 minutes. Results go to a separate folder.
QUICK_TEST = False  #@param {type:"boolean"}

In [ ]:
#@title Paths (no need to change) { display-mode: "form" }
import os, sys, json, shutil, subprocess, glob
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
WORK = Path("/content") if IN_COLAB else Path(os.environ.get("SCRAM_WORK", "colab_work")).resolve()
CODE = WORK / "scram"                  # the project
VIT = CODE / "vision_transformer"
DATA = WORK / "dataset"                # training folder: train/ validation/ test/ <class>/
RAW = WORK / "raw"                     # downloaded datasets
DRIVE = Path(DRIVE_DIR)

if QUICK_TEST:
    MAX_PER_CLASS, EPOCHS_PER_EXPERIENCE, REPLAY_IMAGES_PER_GENERATOR = min(MAX_PER_CLASS, 300), 1, 150
RUN_NAME = f"results_{MODE}" + ("_quicktest" if QUICK_TEST else "")
RESULTS = DRIVE / RUN_NAME             # saved after every class, survives disconnects
CKPT = RESULTS / "replay" / "final_model.pth"

def sh(cmd, cwd=None, env=None):
    """Run a command and stream its output into the notebook (progress bars included)."""
    print("$", " ".join(map(str, cmd)), flush=True)
    full_env = {**os.environ, "PYTHONUNBUFFERED": "1", "TQDM_MININTERVAL": "15", **(env or {})}
    p = subprocess.Popen(list(map(str, cmd)), cwd=cwd, env=full_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    while chunk := p.stdout.read1(4096):
        sys.stdout.write(chunk.decode(errors="replace")); sys.stdout.flush()
    if p.wait():
        raise RuntimeError(f"Command failed (exit code {p.returncode}). See the output above.")

print("Working folder:", WORK, "| results will be saved to:", RESULTS)

## 2. Check the GPU and disk

In [ ]:
import torch
if torch.cuda.is_available():
    GPU = torch.cuda.get_device_name(0)
    print(f"GPU: {GPU} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")
else:
    GPU = None
    print("⚠️  No GPU. Training a ViT on CPU would take days.\n"
          "   Runtime → Change runtime type → T4 GPU, then run the notebook again.")
WORK.mkdir(parents=True, exist_ok=True)
free = shutil.disk_usage(WORK).free / 1e9
print(f"Free disk: {free:.0f} GB")
if free < 60 and (USE_ANIPRO or USE_HUMAN_AI_ARTWORK) and not QUICK_TEST:
    print("⚠️  Less than 60 GB free. AniPrO needs ~15 GB and the Human-AI Artwork set can be large; "
          "if you run out, untick one dataset or lower MAX_PER_CLASS.")

## 3. Connect Google Drive and get the project

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE.mkdir(parents=True, exist_ok=True)

if not (VIT / "art_datasets.py").exists():
    if GITHUB_URL:
        sh(["git", "clone", "--depth", "1", GITHUB_URL, CODE])
    else:
        zip_path = DRIVE / PROJECT_ZIP
        if not zip_path.exists():
            raise FileNotFoundError(f"Upload the project zip to {zip_path} (or set GITHUB_URL).")
        tmp = WORK / "_unzip"
        shutil.unpack_archive(zip_path, tmp)
        top = next(p.parent for p in tmp.rglob("art_datasets.py")).parent   # folder that has vision_transformer/
        shutil.move(str(top), str(CODE))
        shutil.rmtree(tmp, ignore_errors=True)
print("Project ready:", CODE)

## 4. Install what's missing

In [ ]:
sh([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "seaborn", "matplotlib", "tqdm",
    "kagglehub", "huggingface_hub", "pyzipper", "pillow", "numpy"])
if IN_COLAB and not shutil.which("7z"):
    sh(["apt-get", "-qq", "install", "-y", "p7zip-full"])   # fast extraction of AniPrO's encrypted zips
print("Done")

## 5. Sign in to Kaggle and Hugging Face

In [ ]:
def secret(name):
    if os.environ.get(name):
        return os.environ[name]
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    return None

if USE_HUMAN_AI_ARTWORK or USE_REAL_FAKE_ART:
    user, key = secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY")
    if user and key:
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = user, key
        print("Kaggle: using the token from Secrets")
    elif Path("~/.kaggle/kaggle.json").expanduser().exists():
        print("Kaggle: using ~/.kaggle/kaggle.json")
    elif IN_COLAB:
        print("No Kaggle secrets found. Upload your kaggle.json:")
        from google.colab import files
        up = files.upload()
        Path("~/.kaggle").expanduser().mkdir(exist_ok=True)
        dst = Path("~/.kaggle/kaggle.json").expanduser()
        dst.write_bytes(next(iter(up.values()))); dst.chmod(0o600)
    else:
        print("⚠️  No Kaggle credentials; Kaggle downloads will fail.")

token = secret("HF_TOKEN")
if token:
    os.environ["HF_TOKEN"] = token
    print("Hugging Face: using HF_TOKEN")

## 6. Prepare the training folder

In `continue` mode this first unpacks your original dataset (it must contain `train/`, `validation/` and `test/` with the original class folders) and copies your existing model from `BASE_MODEL_DIR` (`final_model.pth` and `metrics.json`).

In [ ]:
DATA.mkdir(parents=True, exist_ok=True)
if MODE == "continue":
    if not (DATA / "train").exists():
        zip_path = DRIVE / BASE_DATASET_ZIP
        if not zip_path.exists():
            raise FileNotFoundError(f"continue mode needs the original dataset at {zip_path}")
        print("Unpacking", zip_path, "...")
        tmp = WORK / "_base"
        shutil.unpack_archive(zip_path, tmp)
        root = next(p.parent for p in tmp.rglob("train") if p.is_dir() and (p.parent / "test").is_dir())
        for split in root.iterdir():
            shutil.move(str(split), str(DATA / split.name))
        shutil.rmtree(tmp, ignore_errors=True)
    if not CKPT.exists():
        base = DRIVE / BASE_MODEL_DIR
        for f in ("final_model.pth", "metrics.json"):
            if not (base / f).exists():
                raise FileNotFoundError(f"continue mode needs {base / f} (from continual_learning/results/replay/)")
        CKPT.parent.mkdir(parents=True, exist_ok=True)
        for f in ("final_model.pth", "metrics.json"):
            shutil.copy2(base / f, CKPT.parent / f)
        print("Copied your existing model to", CKPT.parent)
print("Classes already in the training folder:",
      sorted(p.name for p in (DATA / "train").iterdir()) if (DATA / "train").exists() else "none")

## 7. Download and import the art datasets

For each dataset: download it, show how many images go into each class, then add them. Human art goes to `real`, AI art to a new class. Each dataset is split 80/10/10 into train, validation and test.

If many images are listed as *skipped because their label is unclear*, the dataset's folder names don't say AI or human. Tell me the folder layout and I'll adjust the importer.

In [ ]:
import kagglehub

def imported(source):
    return (DATA / "_imports" / f"{source}.csv").exists()

def import_source(source, src, extra=()):
    cmd = [sys.executable, "art_datasets.py", "import", source, "--src", src, "--out", DATA,
           "--max-per-class", MAX_PER_CLASS, "--no-hint", *extra]
    sh(cmd + ["--dry-run"], cwd=VIT)
    sh(cmd, cwd=VIT)

if USE_ANIPRO and not imported("anipro"):
    dest = RAW / "anipro"
    if not (dest / ".extracted").exists():
        sh([sys.executable, "art_datasets.py", "download", "anipro", "--dest", dest, "--delete-archives"], cwd=VIT)
        (dest / ".extracted").touch()
    import_source("anipro", dest)

for use, source, handle in [
    (USE_HUMAN_AI_ARTWORK, "human-ai-artwork", "adamelkholy/human-ai-artwork"),
    (USE_REAL_FAKE_ART, "real-fake-art", "doctorstrange420/real-and-fake-ai-generated-art-images-dataset"),
]:
    if use and not imported(source):
        print(f"\nDownloading {handle} ...")
        path = kagglehub.dataset_download(handle)
        import_source(source, path)

if ART500K_ZIP and not imported("art500k"):
    dest = RAW / "art500k"
    if not dest.exists():
        print("Unpacking ART500K ...")
        shutil.unpack_archive(DRIVE / ART500K_ZIP, dest)
    import_source("art500k", dest)

# Disk check: the imports are hard links, so the raw downloads can be deleted without losing anything.
print(f"\nFree disk now: {shutil.disk_usage(WORK).free / 1e9:.0f} GB")

## 8. Set up training

In [ ]:
import re

PREFERRED = ["anipro_text2image", "anipro_inpainting", "ai_art_realfake", "ai_artwork", "ai_art_other"]
on_disk = sorted(p.name for p in (DATA / "train").iterdir() if p.is_dir() and p.name != "real")

base_classes, base_generators = ["real"], []
if MODE == "continue":
    ck = torch.load(CKPT, map_location="cpu", weights_only=True)
    base_classes = list(ck["class_names"])
    base_generators = list(ck.get("generators") or [c for c in base_classes if c != "real"])
new = [c for c in on_disk if c not in base_classes]
new.sort(key=lambda c: (PREFERRED.index(c) if c in PREFERRED else 99, c))
CLASSES = base_classes + new
TARGET = new                                               # learned in this order, one experience each
if not new and not CKPT.exists():
    raise RuntimeError("No AI art classes in the training folder. Tick at least one dataset in Settings.")

CONFIG = VIT / "continual_learning" / "cl_common" / "config.py"
MARK = "# ---- set by the Colab notebook ----"

def write_config(generators):
    text = CONFIG.read_text().split(MARK)[0].rstrip() + "\n\n" + MARK + "\n"
    values = {
        "DATA_DIR": str(DATA),
        "CLASS_TO_IDX": {c: i for i, c in enumerate(CLASSES)},
        "NUM_CLASSES": len(CLASSES),
        "GENERATORS": list(generators),
        "MAX_PER_CLASS": 0,
        "BATCH_SIZE": BATCH_SIZE,
        "NUM_WORKERS": 2 if IN_COLAB else 0,
        "EPOCHS_PER_EXPERIENCE": EPOCHS_PER_EXPERIENCE,
        "REPLAY_IMAGES_PER_GENERATOR": REPLAY_IMAGES_PER_GENERATOR,
        "EVAL_SPLIT": "test",
        "RESULTS_DIR": str(RESULTS),
    }
    text += "".join(f"{k} = {v!r}\n" for k, v in values.items())
    text += "CLASS_NAMES = [n for n, _ in sorted(CLASS_TO_IDX.items(), key=lambda kv: kv[1])]\n"
    CONFIG.write_text(text)

def count(split, cls):
    d = DATA / split / cls
    return sum(1 for _ in d.iterdir()) if d.exists() else 0

print(f"{'class':<22}{'train':>8}{'val':>8}{'test':>8}")
for c in CLASSES:
    print(f"{c:<22}{count('train', c):>8}{count('validation', c):>8}{count('test', c):>8}")
    if c in new and count("test", c) == 0:
        print(f"   ⚠️  {c} has no test images, so its accuracy can't be measured")

speed = {"T4": 110, "L4": 220, "A100": 520, "H100": 800}.get(next((k for k in ["H100", "A100", "L4", "T4"] if GPU and k in GPU), ""), 90)
real_n = count("train", "real")
seconds = sum((real_n + 2 * count("train", g)) * EPOCHS_PER_EXPERIENCE / speed for g in TARGET)
print(f"\nPlan: learn {TARGET} one at a time ({len(TARGET)} experiences).")
print(f"Rough training time on {GPU or 'this machine'}: {seconds / 3600:.1f} h (plus evaluation after each class).")

## 9. Train

Each AI class is one experience: the model trains on all human images plus that class, mixed with a replay memory of earlier classes. After each one it is tested on every class learned so far and saved to Drive.

**If Colab disconnects**, reconnect and run all the cells again. Finished classes are skipped.

In [ ]:
def learned():
    if not CKPT.exists():
        return []
    ck = torch.load(CKPT, map_location="cpu", weights_only=True)
    return list(ck.get("generators") or [c for c in ck["class_names"] if c != "real"])

done = learned()
order = (base_generators + TARGET) if MODE == "continue" else TARGET
if done and order[:len(done)] != done:
    raise RuntimeError(f"The saved model learned {done}, which doesn't match the plan {order}. "
                       f"Delete {RESULTS} to start over.")

cl = VIT / "continual_learning"
if not done:
    print(f"\n######## Experience 1/{len(order)}: {order[0]} ########")
    write_config(order[:1])
    sh([sys.executable, cl / "replay" / "run.py"], cwd=VIT)
for i in range(len(learned()) + 1, len(order) + 1):
    print(f"\n######## Experience {i}/{len(order)}: {order[i - 1]} ########")
    write_config(order[:i])
    sh([sys.executable, cl / "replay" / "add_generator.py"], cwd=VIT)
print("\n✅ Training finished. Everything is in", RESULTS / "replay")

## 10. Results

In [ ]:
from IPython.display import Image, display, Markdown

m = json.loads((RESULTS / "replay" / "metrics.json").read_text())
s = m["summary"]
rows = [
    ("AI vs human accuracy", f"{s['final_real_vs_ai_accuracy']:.1%}"),
    ("AI vs human ROC-AUC", f"{s['final_real_vs_ai_auc']:.3f}" if s.get("final_real_vs_ai_auc") else "-"),
    ("AI vs human F1", f"{s['final_real_vs_ai_f1']:.1%}"),
    ("Equal error rate", f"{s['final_real_vs_ai_eer']:.1%}" if s.get("final_real_vs_ai_eer") else "-"),
    ("Exact source accuracy", f"{s['final_accuracy']:.1%}"),
    ("Macro F1 (exact source)", f"{s['final_macro_f1']:.1%}"),
    ("Calibration error (ECE)", f"{s['final_ece']:.3f}"),
    ("Forgetting", f"{(s['final_forgetting'] or 0) * 100:.1f} pts"),
]
display(Markdown("| Metric | Value |\n|---|---|\n" + "\n".join(f"| {a} | {b} |" for a, b in rows)))

last = m["ai_det"][-1]
display(Markdown("**AI vs human accuracy per class (final model)**\n\n| Class | Accuracy |\n|---|---|\n"
                 + "\n".join(f"| {c} | {v:.1%} |" for c, v in last.items())))
print((RESULTS / "replay" / "final_report.txt").read_text())
for png in ["roc_real_vs_ai_final.png", "confusion_matrix_final.png"]:
    if (RESULTS / "replay" / png).exists():
        display(Image(str(RESULTS / "replay" / png), width=640))

## 11. Try it on your own images

In [ ]:
#@markdown Run this cell and choose one or more images. On Colab an upload button appears.
TEST_IMAGES = []  # or list paths, e.g. ["/content/drive/MyDrive/my_painting.jpg"]

os.environ["MODEL_PATH"] = str(CKPT)
sys.path.insert(0, str(CODE))
from backend.inference import Detector
det = Detector(); det.load()
if det.info.error:
    raise RuntimeError(det.info.error)

if not TEST_IMAGES and IN_COLAB:
    from google.colab import files
    TEST_IMAGES = list(files.upload().keys())
for path in TEST_IMAGES:
    r = det.analyze(Path(path).read_bytes())
    verdict = "AI-generated" if r["prediction"] == "AI" else "human-made"
    print(f"\n{Path(path).name}: likely {verdict} ({r['ai_probability']:.0%} AI)")
    for src in r["sources"][:3]:
        print(f"   {src['label']:<38} {src['probability']:.1%}")
    for note in r["notes"]:
        print("   note:", note)
    if IN_COLAB:
        display(Image(path, width=320))

## 12. Use the new model in the app

The model and results are already on your Drive in the results folder. To use them in the website:

1. Download `final_model.pth` and `metrics.json` from that folder (see the path printed below).
2. Put both in `vision_transformer/continual_learning/results/replay/` in your project. Replace the old `metrics.json`.
3. Replace `frontend/src/data/metrics-snapshot.json` with the one this cell saves to Drive.
4. Restart the backend. It loads the newest model automatically.

In [ ]:
target = VIT / "continual_learning" / "results" / "replay"
target.mkdir(parents=True, exist_ok=True)
shutil.copy2(RESULTS / "replay" / "metrics.json", target / "metrics.json")
out = subprocess.run([sys.executable, "-m", "backend.metrics"], cwd=CODE, capture_output=True, text=True, check=True).stdout
(DRIVE / "metrics-snapshot.json").write_text(out)
print("Model:            ", RESULTS / "replay" / "final_model.pth")
print("Metrics:          ", RESULTS / "replay" / "metrics.json")
print("Dashboard snapshot:", DRIVE / "metrics-snapshot.json")